# FTWP Chapter 2 — Review and Exercises

This notebook consolidates the **six non-capstone tasks** for FTWP Chapter 2:
three review exercises and three regular exercises. The problem statements are
paraphrased from the CPF exercise sheet. Numerical parameters are stated where
the source task leaves them open.

The notebook is designed to run from top to bottom in Google Colab or a standard
Python/Jupyter environment.

In [1]:
%matplotlib inline

## Review exercises

### 1. Arrow-Debreu securities

The two Arrow-Debreu payoff vectors are

$$e_u=(1,0),\qquad e_d=(0,1).$$

### 2. Completeness check

Using bond and stock payoffs, form $M=[B_1\;S_1]$ and check whether it is
invertible.

### 3. Replication and price

For a claim $C_1$, solve $M\phi=C_1$ and compute $C_0=M_0^\top\phi$.

In [2]:
import numpy as np
import pandas as pd

states = ["Up", "Down"]
e_up = np.array([1.0, 0.0])
e_down = np.array([0.0, 1.0])

B0, S0 = 10.0, 10.0
B1 = np.array([11.0, 11.0])
S1 = np.array([20.0, 5.0])
M0 = np.array([B0, S0])
M = np.column_stack((B1, S1))

C1 = np.array([5.0, 0.0])
phi = np.linalg.solve(M, C1)
C0 = M0 @ phi

print("Arrow-Debreu Up payoff:", e_up)
print("Arrow-Debreu Down payoff:", e_down)
print("det(M):", np.linalg.det(M))
print("rank(M):", np.linalg.matrix_rank(M))
print("Replication holdings:", phi)
print("Replicated payoff:", M @ phi)
print("Arbitrage-free price:", C0)

Arrow-Debreu Up payoff: [1. 0.]
Arrow-Debreu Down payoff: [0. 1.]
det(M): -165.00000000000009
rank(M): 2
Replication holdings: [-0.15151515  0.33333333]
Replicated payoff: [5.0000000e+00 4.4408921e-16]
Arbitrage-free price: 1.8181818181818183


The matrix has rank two and is invertible, so every two-state payoff can be
replicated uniquely in this model.

## Regular exercises

### 4. State prices

The state-price vector $\psi$ satisfies

$$M_0=M^\top\psi.$$

### 5. Risk-neutral probabilities

The one-period rate is $r=10\%$, so $\sum_s\psi_s=1/(1+r)$. The risk-neutral
probabilities are $q_s=(1+r)\psi_s$.

### 6. A too-cheap claim and an explicit arbitrage

Suppose the claim trades for $1.50$ even though its replicating portfolio costs
$20/11$. Buy one claim and short the replicating portfolio. Future cashflows
cancel, while the initial receipt is positive.

In [3]:
psi = np.linalg.solve(M.T, M0)
r = B1[0] / B0 - 1.0
q = (1.0 + r) * psi

price_state_prices = psi @ C1
price_risk_neutral = (q @ C1) / (1.0 + r)

too_cheap_price = 1.50
claim_units = 1.0
asset_holdings = -phi  # short the replicating portfolio
initial_receipt = -(too_cheap_price + M0 @ asset_holdings)
future_net_payoff = claim_units * C1 + M @ asset_holdings

summary = pd.DataFrame(
    {
        "State price": psi,
        "Risk-neutral probability": q,
        "Claim payoff": C1,
    },
    index=states,
)

display(summary)
print("M^T psi:", M.T @ psi)
print("State-price valuation:", price_state_prices)
print("Risk-neutral valuation:", price_risk_neutral)
print("Initial arbitrage receipt:", initial_receipt)
print("Future net payoff:", future_net_payoff)

,State price,Risk-neutral probability,Claim payoff
Up,0.363636,0.4,5.0
Down,0.545455,0.6,0.0


M^T psi: [10. 10.]
State-price valuation: 1.8181818181818186
Risk-neutral valuation: 1.8181818181818183
Initial arbitrage receipt: 0.31818181818181834
Future net payoff: [ 0.0000000e+00 -4.4408921e-16]


The strategy has a strictly positive initial receipt and zero future payoff in
both states. That is an arbitrage because the claim was quoted below the cost of
an exactly equivalent traded-asset portfolio.

In [4]:
checks = {
    "complete market": np.linalg.matrix_rank(M) == 2,
    "replication": np.allclose(M @ phi, C1),
    "state-price restriction": np.allclose(M.T @ psi, M0),
    "positive state prices": np.all(psi > 0),
    "risk-neutral probabilities": np.all(q > 0) and np.isclose(q.sum(), 1.0),
    "pricing identity": np.isclose(C0, price_state_prices) and np.isclose(C0, price_risk_neutral),
    "arbitrage initial gain": initial_receipt > 0,
    "arbitrage future payoff": np.allclose(future_net_payoff, 0.0),
}

for name, passed in checks.items():
    print(f"{name}: {passed}")
assert all(checks.values())

complete market: True
replication: True
state-price restriction: True
positive state prices: True
risk-neutral probabilities: True
pricing identity: True
arbitrage initial gain: True
arbitrage future payoff: True
